In [1]:
import pandas as pd

In [2]:
df=pd.read_csv(r"D:\CODING\PYTHON\NLP\9.LSTM\fake_or_real_news.csv")

In [5]:
df.head()

,Unnamed: 0,title,text,label
0,8476,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,10294,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,3608,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,10142,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,875,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [6]:
df.shape

(6335, 4)

In [7]:
df.drop(columns=["Unnamed: 0"],inplace=True)

In [8]:
df.isnull().sum()

title    0
text     0
label    0
dtype: int64

In [10]:
df.head()

,title,text,label
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello...",FAKE
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...,FAKE
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...,REAL
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T...",FAKE
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...,REAL


In [11]:
from sklearn.preprocessing import LabelEncoder

la=LabelEncoder()
df["label"]=la.fit_transform(df["label"])

In [12]:
## Get the Independent Features

X=df.drop('label',axis=1)

In [13]:
## Get the Dependent features
y=df['label']

In [14]:
X.shape

(6335, 2)

In [15]:
y.shape

(6335,)

In [16]:
import tensorflow as tf

In [17]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [18]:
### Vocabulary size
voc_size=5000

### Onehot Representation

In [19]:
messages=X.copy()

In [20]:
messages['title'][1]

'Watch The Exact Moment Paul Ryan Committed Political Suicide At A Trump Rally (VIDEO)'

In [21]:
messages

,title,text
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello..."
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T..."
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...
...,...,...
6330,State Department says it can't find emails fro...,The State Department told the Republican Natio...
6331,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...
6332,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...
6333,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene..."


In [27]:
#messages.reset_index(inplace=True)

In [22]:
messages

,title,text
0,You Can Smell Hillary’s Fear,"Daniel Greenfield, a Shillman Journalism Fello..."
1,Watch The Exact Moment Paul Ryan Committed Pol...,Google Pinterest Digg Linkedin Reddit Stumbleu...
2,Kerry to go to Paris in gesture of sympathy,U.S. Secretary of State John F. Kerry said Mon...
3,Bernie supporters on Twitter erupt in anger ag...,"— Kaydee King (@KaydeeKing) November 9, 2016 T..."
4,The Battle of New York: Why This Primary Matters,It's primary day in New York and front-runners...
...,...,...
6330,State Department says it can't find emails fro...,The State Department told the Republican Natio...
6331,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...,The ‘P’ in PBS Should Stand for ‘Plutocratic’ ...
6332,Anti-Trump Protesters Are Tools of the Oligarc...,Anti-Trump Protesters Are Tools of the Oligar...
6333,"In Ethiopia, Obama seeks progress on peace, se...","ADDIS ABABA, Ethiopia —President Obama convene..."


In [23]:
import nltk
import re
from nltk.corpus import stopwords

In [24]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sinha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [26]:
### Dataset Preprocessing
from nltk.stem.porter import PorterStemmer ##stemming purpose
ps = PorterStemmer()
corpus = []
for i in range(0, len(messages)):
    print(i)
    
    review = re.sub('[^a-zA-Z]', ' ', messages['title'][i])
    review = review.lower()
    review = review.split()

    review = [ps.stem(word) for word in review if not word in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [27]:
corpus

['smell hillari fear',
 'watch exact moment paul ryan commit polit suicid trump ralli video',
 'kerri go pari gestur sympathi',
 'berni support twitter erupt anger dnc tri warn',
 'battl new york primari matter',
 'tehran usa',
 'girl horrifi watch boyfriend left facetim',
 'britain schindler die',
 'fact check trump clinton command chief forum',
 'iran reportedli make new push uranium concess nuclear talk',
 'three clinton iowa glimps fire elud hillari clinton campaign',
 'donald trump shockingli weak deleg game somehow got even wors',
 'strong solar storm tech risk today news oct video',
 'way america prepar world war',
 'trump take cruz lightli',
 'women lead differ',
 'shock michel obama hillari caught glamor date rape promot',
 'hillari clinton huge troubl america notic sick thing hidden pictur liberti writer news',
 'iran bill obama like',
 'chart explain everyth need know partisanship america',
 'slipperi slope trump propos ban muslim',
 'episod sunday wire hail deplor special g

In [28]:
corpus[1]

'watch exact moment paul ryan commit polit suicid trump ralli video'

In [29]:
onehot_repr=[one_hot(words,voc_size)for words in corpus]
onehot_repr

[[3117, 2108, 2020],
 [3402, 1145, 674, 625, 186, 208, 4753, 2621, 3245, 4418, 4823],
 [1335, 1105, 1197, 4661, 3997],
 [1432, 4013, 3044, 2210, 4252, 418, 4059, 3430],
 [4067, 39, 1905, 1429, 1075],
 [1057, 1354],
 [958, 107, 3402, 1343, 4447, 1482],
 [4335, 1885, 4814],
 [1183, 2145, 3245, 718, 2488, 321, 4496],
 [2668, 1779, 1189, 39, 1299, 3408, 3425, 1281, 2724],
 [2581, 718, 443, 635, 3531, 16, 2108, 718, 2424],
 [490, 3245, 4460, 4116, 961, 2414, 2918, 3055, 3012, 146],
 [3653, 468, 3010, 2176, 2915, 2193, 531, 3673, 4823],
 [1358, 1246, 394, 4697, 1630],
 [3245, 4551, 110, 2537],
 [170, 164, 4058],
 [4607, 1425, 2375, 2108, 2166, 1963, 4264, 2238, 850],
 [2108, 718, 4898, 4305, 1246, 4051, 472, 3437, 2069, 715, 1776, 34, 531],
 [2668, 4350, 2375, 2561],
 [2341, 2603, 189, 277, 241, 128, 1246],
 [3081, 110, 3245, 1215, 1396, 3951],
 [4048, 2175, 3747, 3552, 3306, 50, 4787, 377, 2653],
 [2108, 718, 1189, 1856, 850, 2175, 1699],
 [39, 3458, 2691, 4723, 1341, 2580, 2631, 1829],
 [4

In [30]:
corpus[1]

'watch exact moment paul ryan commit polit suicid trump ralli video'

In [31]:
onehot_repr[1]

[3402, 1145, 674, 625, 186, 208, 4753, 2621, 3245, 4418, 4823]

### Embedding Representation

In [32]:
sent_length=20
embedded_docs=pad_sequences(onehot_repr,padding='post',maxlen=sent_length)
print(embedded_docs)

[[3117 2108 2020 ...    0    0    0]
 [3402 1145  674 ...    0    0    0]
 [1335 1105 1197 ...    0    0    0]
 ...
 [2044 3245 3039 ...    0    0    0]
 [4838 2375 3935 ...    0    0    0]
 [3181  549 2819 ...    0    0    0]]


In [33]:
embedded_docs[1]

array([3402, 1145,  674,  625,  186,  208, 4753, 2621, 3245, 4418, 4823,
          0,    0,    0,    0,    0,    0,    0,    0,    0])

In [34]:
embedded_docs[0]

array([3117, 2108, 2020,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0])

In [36]:
## Creating model
embedding_vector_features=40 ##features representation

model=Sequential()

model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(LSTM(100))

model.add(Dense(1,activation='sigmoid'))


model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])
print(model.summary())

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [37]:
len(embedded_docs),y.shape

(6335, (6335,))

In [38]:
import numpy as np
X_final=np.array(embedded_docs)
y_final=np.array(y)

In [39]:
X_final.shape,y_final.shape

((6335, 20), (6335,))

In [40]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.33, random_state=42)

### Model Training

In [42]:
### Finally Training
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=32)

Epoch 1/10
  1/133 ━━━━━━━━━━━━━━━━━━━━ 7s 59ms/step - accuracy: 1.0000 - loss: 0.0151

133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9651 - loss: 0.1088 - val_accuracy: 0.7480 - val_loss: 0.9098
Epoch 2/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9607 - loss: 0.1129 - val_accuracy: 0.7456 - val_loss: 1.0174
Epoch 3/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9689 - loss: 0.1043 - val_accuracy: 0.7370 - val_loss: 0.9362
Epoch 4/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9771 - loss: 0.0754 - val_accuracy: 0.7499 - val_loss: 1.0131
Epoch 5/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.9847 - loss: 0.0622 - val_accuracy: 0.7480 - val_loss: 1.1055
Epoch 6/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9870 - loss: 0.0524 - val_accuracy: 0.7370 - val_loss: 0.9603
Epoch 7/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9840 - loss: 0.0622 - val_accuracy: 0.7394 - val_loss: 1.0033
Epoch 8/10
133/133 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.9868 - loss: 0.0542 - val_accuracy: 0.742

In [44]:
y_pred=model.predict(X_test)

66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [45]:
y_pred>=0.5

array([[False],
       [False],
       [False],
       ...,
       [ True],
       [False],
       [ True]])

In [46]:
y_pred=np.where(y_pred > 0.6, 1,0) ##AUC ROC Curve

In [48]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

array([[765, 306],
       [248, 772]], dtype=int64)

In [51]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.7350549976087997

### Adding Dropout

In [52]:
from tensorflow.keras.layers import Dropout
## Creating model
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sent_length))
model.add(Dropout(0.3))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


### Performance Metrics And Accuracy

In [53]:
y_pred=model.predict(X_test)

66/66 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step


In [54]:
y_pred=np.where(y_pred > 0.6, 1,0) ##AUC ROC Curve

In [55]:
from sklearn.metrics import confusion_matrix


In [56]:
confusion_matrix(y_test,y_pred)

array([[1071,    0],
       [1020,    0]], dtype=int64)

In [57]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.5121951219512195

In [58]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.51      1.00      0.68      1071
           1       0.00      0.00      0.00      1020

    accuracy                           0.51      2091
   macro avg       0.26      0.50      0.34      2091
weighted avg       0.26      0.51      0.35      2091



d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
d:\CODING\PYTHON\PYTHON---3.11\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
